# Формирование датасета, EDA и очистка

## Импортируем библиотеки и создаем полезные функции

In [29]:
import pandas as pd
from dateutil.relativedelta import relativedelta

In [30]:
# Функция для замены значения на ключ словаря
def replace_with_dict_key(value, rep_dict):
    for key, values_list in rep_dict.items():
        if value in values_list:
            return key
    return None 

## Загружаем данные

In [31]:
raw_df = pd.read_csv(r'data/undefind_DSMED.csv', sep=';')
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36297 entries, 0 to 36296
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID истории болезни                    36297 non-null  object 
 1   Осн. диаг. при выписке МКБ10 (текст)  36297 non-null  object 
 2   Заголовок документа                   36297 non-null  object 
 3   Кол. лаб. показатель                  36236 non-null  object 
 4   Значение кол. показателя              36236 non-null  float64
 5   Ед. изм. кол. показателя              36236 non-null  object 
 6   Норма кол. показателя                 36236 non-null  object 
 7   Флаг нормы кол. показателя            36236 non-null  object 
 8   Кач. лаб. показатель                  7005 non-null   object 
 9   Значение кач. показателя              7005 non-null   object 
 10  Норма кач. показателя                 7005 non-null   object 
 11  Пол            

## Собственно, формирование датасета

In [32]:
# Посмотрим количество уникальных значений по колонкам
raw_df.nunique()

ID истории болезни                       243
Осн. диаг. при выписке МКБ10 (текст)      11
Заголовок документа                        1
Кол. лаб. показатель                     131
Значение кол. показателя                2608
Ед. изм. кол. показателя                  11
Норма кол. показателя                    123
Флаг нормы кол. показателя                 4
Кач. лаб. показатель                       7
Значение кач. показателя                 155
Норма кач. показателя                      5
Пол                                        2
Дата рождения пациента                   104
dtype: int64

Так как прогнозировать будем по показателям, то наши признаки в эталонном датасете - показатели лабораторных тестов

In [33]:
# Посмотрим состав Кол. лабораторных показателей
raw_df['Кол. лаб. показатель'].value_counts()

Кол. лаб. показатель
Гемоглобин (HGB)                                        1114
Средний объем эритроцита (MCV)                          1114
Гематокрит (HCT)                                        1114
Среднее содержание гемоглобина в эритроците (MCH)       1114
Средняя концентрация гемоглобина в эритроците (MCHC)    1114
                                                        ... 
Тромбокрит                                                 1
Эозинофилы, абсолютное количество                          1
Базофилы, абсолютное количество                            1
СОЭ по Панченкову                                          1
MXD#                                                       1
Name: count, Length: 131, dtype: int64

In [34]:
replacement_dict = {
    'wbc': ['Лейкоциты', 'Лейкоциты (WBC)', 'Общее количество лейкоцитов (WBC)', 'WBC'],
    'rbc': ['Эритроциты', 'Эритроциты (RBC)', 'Общее количество эритроцитов (RBC)', 'RBC'],
    'hgb': ['Гемоглобин', 'Гемоглобин (HGB)', 'HGB'],
    'hct': ['Гематокрит', 'Гематокрит (HCT)', 'HCT'],
    'mcv': ['Средний объем эритроцита', 'Средний объем эритроцита (MCV)', 'MCV'],
    'mch': ['Среднее содержание гемоглобина в эритроците', 'Среднее содержание гемоглобина в эритроците (MCH)', 'MCH'],
    'mchc': ['Средняя концентрация гемоглобина в эритроците', 'Средняя концентрация гемоглобина в эритроците (MCHC)', 'MCHC'],
    'plt': ['Тромбоциты', 'Тромбоциты (PLT)', 'PLT'],
    'pct': ['Тромбокрит', 'Тромбокрит (PCT)', 'PCT'],
    'mpv': ['Средний объем тромбоцита', 'Средний объём тромбоцитов', 'Средний объем тромбоцита (MPV)', 'MPV (Средний объём тромбоцитов)', 'MPV ', 'MPV'],
    'pdw': ['Ширина распределения тромбоцитов', 'Ширина распределения тромбоцитов по объему', 'Ширина распределения тромбоцитов (PDW)', 'PDW'],
    'rdw': ['Ширина распределения эритроцитов', 'Ширина распределения эритроцитов по объему (RDW)', 'RDW', 'Ширина распределения эритроцитов (RDW)'],
    'rdv_cv': ['Ширина распределения эритроцитов по объему, коэффициент вариации (RDW-CV)', 'Ширина распределения эритроцитов по объему, коэффициент вариации', 'RDW-CV '],
    'rdv_sd': ['Ширина распределения эритроцитов по объему, стандартное отклонение (RDW-SD)','Ширина распределения эритроцитов по объему, стандартное отклонение','Ширина распределения эритроцитов, стандартное отклонение (RDW-SD)', 'RDW-SD'],
    'ne_abs': ['Нейтрофилы, абсолютное количество', 'Нейтрофилы, абсолютное количество (NE#)', 'Абсолютное количество нейтрофилов (NE#)', 'Нейтрофилы #'],
    'ne_rel': ['Нейтрофилы, относительное количество', 'Нейтрофилы, относительное количество (NE%)', 'Относительное количество нейтрофилов (NE%)', 'Нейтрофилы %', 'NE%'],
    'ly_abs': ['Лимфоциты, абсолютное количество', 'Лимфоциты, абсолютное количество (LY#)', 'Лимфоциты', 'Абсолютное количество лимфоцитов (LY#)', 'Лимфоциты #'],
    'ly_rel': ['Лимфоциты, относительное количество', 'Лимфоциты, относительное количество (LY%)', 'Лимфоциты %', 'Относительное количество лимфоцитов (LY%)', 'Лимфоциты %', 'LY%'],
    'mo_abs': ['Моноциты, абсолютное количество', 'Моноциты, абсолютное количество (MO#)', 'Моноциты', 'Абсолютное количество моноцитов (MO#)', 'Моноциты #'],
    'mo_rel': ['Моноциты, относительное количество', 'Моноциты, относительное количество (MO%)', 'Моноциты %', 'Моноциты %', 'Относительное количество моноцитов (MO%)', 'MO%'],
    'eo_abs': ['Эозинофилы, абсолютное количество', 'Эозинофилы, абсолютное количество (EO#)', 'Эозинофилы', 'Абсолютное количество эозинофилов (EO#)', 'Эозинофилы # '],
    'eo_rel': ['Эозинофилы, относительное количество', 'Эозинофилы, относительное количество (EO%)', 'Эозинофилы %', 'Относительное количество эозинофилов (EO%)', 'EO%'],
    'ba_abs': ['Базофилы, абсолютное количество', 'Базофилы, абсолютное количество (BA#)', 'Базофилы', 'Абсолютное количество базофилов (BA#)', 'Базофилы #', 'Базофилы # '],
    'ba_rel': ['Базофилы, относительное количество', 'Базофилы, относительное количество (BA%)', 'Базофилы %', 'Относительное количество базофилов (BA%)', 'BA%'],
    'mxd_abs': ['Смешанная фракция, абсолютное количество (MXD#)', 'MXD# ', 'Смешанная фракция, абсолютное количество', 'MXD#', 'MXD'],
    'mxd_rel': ['Смешанная фракция, относительное количество (MXD%)', 'MXD%', 'Смешанная фракция, относительное количество'],
    'soe': ['СОЭ Вест.', 'Скорость оседания эритроцитов (СОЭ) по Вестергрену', 'СОЭ по Панченкову', 'СОЭ Панч.'],
    'cp': ['Цветовой показатель'],
    'pal': ['Палочкоядерные'],
    'seg': ['Сегментоядерные'],
    'plasma': ['Плазматические клетки', 'Плазматич. клетки'],
    'myelo': ['Миелоциты'],
    'yunye': ['Юные'],
    'blasty': ['Бласты'],
    'normobl_abs': ['Нормобласты', 'Нормобласты #'],
    'normobl_rel': ['Нормобласты %'],
    'ret_rel': ['RET%', 'Ретикулоциты %'],
    'ret_abs': ['Ретикулоциты кол-во'],
    'plcr': ['P-LCR'],
    'noclass_abs': ['Неклассифицируемые кол-во'],
    'noclass_rel': ['Неклассифицируемые %'],
    'prolym': ['Пролимфоциты'],
    'promyelo': ['Промиелоциты']
}

В составе тестов увидели много синонимов - нужно мапить на уникальные признаки. Сформировали словарь для маппинга

In [35]:
# начинаем первую очистку и создание нужных признаков
prep_df = raw_df.copy()
# Удаляем строки, где вообще нет показателей
# Конечно мы потеряем 61 строку с комментариями, но они либо о браке, либо непонятные
prep_df = prep_df.dropna(subset=['Кол. лаб. показатель'])
# Посчитаем возраст пациента в годах на 21.07.2025 - начало хакатона. Других дат нет
age_date = pd.to_datetime('2025-07-21')
prep_df['Дата рождения пациента'] = pd.to_datetime(prep_df['Дата рождения пациента'])
prep_df['age'] = prep_df['Дата рождения пациента'].apply(lambda x: relativedelta(age_date, x).years)
# Закодируем пол числом
prep_df['gender'] = prep_df['Пол'].apply(lambda x: 0 if 'ж' in x.lower() else 1)


In [36]:
# Формируем номер лаб исследования исходя из логики, что тесты в датасете расположены
# по хронологии сверху вниз и в одном ЛИ тесты не повторяются
lab_study = []
study_prefix = 0

def create_lab_study_num(lab_test):

    global lab_study
    global study_prefix
    res = 'result_'

    if lab_test in lab_study:
        lab_study.clear()
        study_prefix += 1
    else:
        lab_study.append(lab_test)
    return res + str(study_prefix)
prep_df['study_num'] = prep_df['Кол. лаб. показатель'].apply(lambda x: create_lab_study_num(x))
# формируем уникальный ID ЛИ из ID истории и номера ЛИ
prep_df['study_ID'] = prep_df['ID истории болезни'] + '_' + prep_df['study_num']

In [37]:
# добавляем унифицированные коды тестов из словаря синонимов
prep_df['lab_test'] = prep_df['Кол. лаб. показатель'].apply(lambda x: replace_with_dict_key(x, replacement_dict))

In [38]:
# посмотрим сколько тестов осталось после унификации
prep_df['lab_test'].value_counts()

lab_test
ly_abs         2213
hgb            1619
rbc            1619
mch            1619
mcv            1619
mchc           1619
hct            1619
plt            1617
wbc            1616
mo_abs         1518
ly_rel         1470
eo_abs         1388
ne_rel         1332
ne_abs         1319
ba_abs         1087
mpv             982
pct             925
cp              832
rdw             829
eo_rel          824
ba_rel          822
mo_rel          815
soe             794
seg             740
pal             736
rdv_sd          698
rdv_cv          696
pdw             657
mxd_abs         566
mxd_rel         468
myelo           316
yunye           304
noclass_rel     239
normobl_abs     192
blasty          143
noclass_abs     117
plcr             81
ret_rel          57
ret_abs          44
plasma           30
promyelo         21
normobl_rel      18
prolym           16
Name: count, dtype: int64

In [39]:
# Сформируем справочный фрейм с данными пациенов, чтоб домержить в основной датасет
pasp_df = prep_df[['study_ID', 'gender', 'age']].drop_duplicates().copy()
pasp_df

,study_ID,gender,age
0,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_0,0,62
16,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_1,0,62
47,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_2,0,62
78,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_3,0,62
107,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_4,0,62
...,...,...,...
36176,6166e156-c1af-11ed-8602-005056880ecb_result_2637,1,43
36203,6166e156-c1af-11ed-8602-005056880ecb_result_2638,1,43
36227,6166e156-c1af-11ed-8602-005056880ecb_result_2639,1,43
36253,6166e156-c1af-11ed-8602-005056880ecb_result_2640,1,43


In [40]:
# Проверим референсные значения
ref_df = prep_df[['Пол', 'lab_test', 'Норма кол. показателя']].drop_duplicates()
# ref_df = prep_df[['Пол', 'Кол. лаб. показатель', 'Норма кол. показателя']].drop_duplicates()

ref_df[['lo_ref', 'hi_ref']] = ref_df['Норма кол. показателя'].str.split(':', n=1, expand=True)
ref_df.drop(columns=['Норма кол. показателя'], inplace=True)
ref_df['lo_ref'] = ref_df['lo_ref'].str.replace(',', '.')
ref_df['hi_ref'] = ref_df['hi_ref'].str.replace(',', '.')
ref_df.dropna(inplace=True)
ref_df[['lo_ref', 'hi_ref']] = ref_df[['lo_ref', 'hi_ref']].astype('float')
ref_df.drop_duplicates(inplace=True)
ref_df.sort_values(['Пол', 'lab_test'])
# ref_df.sort_values(['Пол', 'Кол. лаб. показатель'])


,Пол,lab_test,lo_ref,hi_ref
37,Ж,ba_abs,0.0,0.10
129,Ж,ba_abs,0.0,2.00
32,Ж,ba_rel,0.0,2.00
308,Ж,blasty,0.0,0.00
38,Ж,cp,0.8,1.05
...,...,...,...,...
484,М,soe,2.0,20.00
3590,М,soe,1.0,20.00
462,М,wbc,4.0,11.00
11344,М,wbc,4.0,8.80


Видим, что референсные значения разные для одних и тех же тестов, даже для одного пола. Похоже, на них не сможем ориентироваться, будем ориентироваться на флаги результата, т.к. они индивидуальны для каждого теста.

In [41]:
# Закодируем флаг результата числом: норма = 0, повышенный = 1, пониженный = -1
# пригодится, если захотим обучать не на абсолютах, а по флагам результата
prep_df['result'] = prep_df['Флаг нормы кол. показателя'].apply(lambda x: 0 if 'норм' in x.lower()
                                                                else (-1 if "пониж" in x.lower()
                                                                      else (1 if "повыш" in x.lower()
                                                                            else x)))

Займемся качественными показателями. У них нет числового результата и результатом будем считать флаг наличия или отсутствич признака.
Часть качественных признаков названа явно, а часть их описаны в комментариях, не все из которых полезны.

In [42]:
# Выделяем кач показатели, которые не в комментариях
mask = prep_df['Кач. лаб. показатель'].notna() & prep_df['Кач. лаб. показатель'].ne('Комментарий')
real_qual_df = prep_df[mask][['study_ID', 'Кач. лаб. показатель', 'Значение кач. показателя', 'Норма кач. показателя']].copy()
real_qual_df

,study_ID,Кач. лаб. показатель,Значение кач. показателя,Норма кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6_result_38,Анизоцитоз,+,+-
931,76416b20-b70b-11ec-ab54-0050568844e6_result_39,Гипохромия,+,+-
932,76416b20-b70b-11ec-ab54-0050568844e6_result_39,Анизоцитоз,+,+-
933,76416b20-b70b-11ec-ab54-0050568844e6_result_40,Гипохромия,+,+-
934,76416b20-b70b-11ec-ab54-0050568844e6_result_40,Анизоцитоз,+,+-
...,...,...,...,...
36037,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,Пойкилоцитоз,+++,+-
36039,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,Анизоцитоз,+++,+-
36040,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,Пойкилоцитоз,+++,+-
36042,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,Анизоцитоз,+++,+-


In [43]:
# Посмотрим на значения нормы
real_qual_df['Норма кач. показателя'].value_counts(dropna=False)

Норма кач. показателя
+-             2170
Общая норма     223
0:1              29
0:2              20
Name: count, dtype: int64

In [44]:
# Посмотрим варианты нормы для разных кач показателей
real_qual_df.groupby(['Кач. лаб. показатель', 'Норма кач. показателя']).count()

study_ID  Значение кач. показателя
Кач. лаб. показатель Норма кач. показателя                                    
Анизоцитоз           +-                          910                       910
                     Общая норма                 113                       113
Гипохромия           +-                          191                       191
                     Общая норма                  54                        54
Макроцитоз           +-                          141                       141
Микроцитоз           +-                          139                       139
                     Общая норма                  29                        29
Нормобласты          0:1                          29                        29
                     0:2                          20                        20
Пойкилоцитоз         +-                          789                       789
                     Общая норма                  27                        27

In [45]:
# Посмотрим на реальные значения кач показателей
real_qual_df['Значение кач. показателя'].value_counts(dropna=False)

Значение кач. показателя
+            1250
++            895
+++           182
+-             66
3.0:100.0      29
3:100          20
Name: count, dtype: int64

In [46]:
# Сформируем словарь для кодирования результатов
real_result_dict = {
    0: ['+', '-', '+-'],
    1: ['++', '+++', '3.0:100.0', '3:100']
}
# Сформируем словарь для перекодирования наименований показателей (задалбывает переключать раскладку)
rus_to_eng_dict = {
    'anisocytos': ['Анизоцитоз'],
    'hypochromia': ['Гипохромия'],
    'macrocytos': ['Макроцитоз'],
    'microcytos': ['Микроцитоз'],
    'poikilocytos': ['Пойкилоцитоз'],
    'normoblast': ['Нормобласты']
}
# Перекодируем результаты числами
real_qual_df['Значение кач. показателя'] = real_qual_df['Значение кач. показателя'].apply(lambda x: replace_with_dict_key(x, real_result_dict))
real_qual_df['Кач. лаб. показатель'] = real_qual_df['Кач. лаб. показатель'].apply(lambda x: replace_with_dict_key(x, rus_to_eng_dict))
real_qual_df.drop(columns=['Норма кач. показателя'], inplace=True)
real_qual_df.drop_duplicates()

,study_ID,Кач. лаб. показатель,Значение кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6_result_38,anisocytos,0
931,76416b20-b70b-11ec-ab54-0050568844e6_result_39,hypochromia,0
932,76416b20-b70b-11ec-ab54-0050568844e6_result_39,anisocytos,0
933,76416b20-b70b-11ec-ab54-0050568844e6_result_40,hypochromia,0
934,76416b20-b70b-11ec-ab54-0050568844e6_result_40,anisocytos,0
...,...,...,...
36037,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,poikilocytos,1
36039,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,anisocytos,1
36040,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,poikilocytos,1
36042,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,anisocytos,1


In [47]:
# переведем значения кач показателей из строк в столбцы
work_real_qual_df = real_qual_df.pivot_table(index='study_ID', columns='Кач. лаб. показатель', values='Значение кач. показателя', aggfunc='first')

Предлагаю комментарии в признаки не добавлять. Работы куча, а толку - ноль.

In [48]:
# # Выделяем кач показатели, которые в комментариях
# mask = prep_df['Кач. лаб. показатель'] == 'Комментарий'
# real_comm_df = prep_df[mask][['study_ID', 'Значение кач. показателя']].copy()
# real_comm_df

In [49]:
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('проверен', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('Проверен', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('ПРОВЕРЕН', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромб.пров', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромбоциты по мазку', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромбоциты соответствуют', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('невозм', na=False)]

In [50]:
# real_comm_df[real_comm_df['Значение кач. показателя'].str.contains('промоноцит', na=False)] = 'promonocit'
# real_comm_df[real_comm_df['Значение кач. показателя'].str.contains('полихроматофильные эритроциты', na=False)] = 'polychrome_rbc'

In [51]:
# real_comm_df['Значение кач. показателя'].value_counts()

In [52]:
# Соберем датасет для работы
work_df = prep_df.pivot_table(values='Значение кол. показателя', columns='lab_test', index='study_ID', aggfunc='first')
work_df = work_df.merge(pasp_df, on='study_ID', how='left')
work_df = work_df.merge(work_real_qual_df, on='study_ID', how='left')

In [53]:
# Это код для формирования датасета для обучения на основе флагов результатов
# flag_work_df = prep_df.pivot_table(values='result', columns='lab_test', index='study_ID', aggfunc='first')
# flag_work_df = flag_work_df.merge(pasp_df, on='study_ID', how='left')
# flag_work_df = flag_work_df.merge(work_real_qual_df, on='study_ID', how='left')

## EDA

In [54]:
display(work_df.info())
display(work_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3115 entries, 0 to 3114
Data columns (total 52 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   study_ID      3115 non-null   object 
 1   ba_abs        791 non-null    float64
 2   ba_rel        800 non-null    float64
 3   blasty        139 non-null    float64
 4   cp            802 non-null    float64
 5   eo_abs        867 non-null    float64
 6   eo_rel        803 non-null    float64
 7   hct           1530 non-null   float64
 8   hgb           1506 non-null   float64
 9   ly_abs        1553 non-null   float64
 10  ly_rel        1423 non-null   float64
 11  mch           1551 non-null   float64
 12  mchc          1558 non-null   float64
 13  mcv           1549 non-null   float64
 14  mo_abs        925 non-null    float64
 15  mo_rel        795 non-null    float64
 16  mpv           958 non-null    float64
 17  mxd_abs       512 non-null    float64
 18  mxd_rel       461 non-null  

None

,ba_abs,ba_rel,blasty,cp,eo_abs,eo_rel,hct,hgb,ly_abs,ly_rel,...,wbc,yunye,gender,age,anisocytos,hypochromia,macrocytos,microcytos,normoblast,poikilocytos
count,791.000000,800.000000,139.000000,802.000000,867.000000,803.000000,1530.000000,1506.000000,1553.000000,1423.000000,...,1477.000000,291.000000,3115.000000,3115.000000,970.000000,245.000000,141.0,168.000000,5.0,784.000000
mean,0.379431,1.413550,12.320144,0.856222,0.496494,1.779851,31.756667,105.771116,4.433625,28.186114,...,9.891462,3.303093,0.518780,72.256822,0.632990,0.110204,0.0,0.196429,1.0,0.438776
std,1.074858,1.320643,26.249500,0.118955,1.359203,1.579101,10.613300,31.252131,9.690505,18.950051,...,18.695870,3.502761,0.499727,10.903332,0.482238,0.313785,0.0,0.398484,0.0,0.496554
min,0.000000,0.000000,0.000000,0.490000,0.000000,0.000000,9.000000,34.000000,0.100000,0.000000,...,0.000000,0.000000,0.000000,10.000000,0.000000,0.000000,0.0,0.000000,1.0,0.000000
25%,0.040000,0.600000,1.000000,0.820000,0.030000,0.500000,24.000000,85.000000,0.940000,15.100000,...,3.810000,1.000000,0.000000,68.000000,0.000000,0.000000,0.0,0.000000,1.0,0.000000
50%,0.100000,1.100000,2.000000,0.860000,0.140000,1.500000,30.000000,102.000000,1.500000,23.000000,...,6.640000,2.000000,1.000000,71.000000,1.000000,0.000000,0.0,0.000000,1.0,0.000000
75%,0.225000,1.900000,5.000000,0.900000,0.370000,2.700000,37.900000,124.000000,2.400000,33.250000,...,10.300000,5.000000,1.000000,77.000000,1.000000,0.000000,0.0,0.000000,1.0,1.000000
max,13.000000,13.700000,98.000000,1.350000,22.000000,10.900000,71.700000,220.000000,137.900000,100.000000,...,294.770000,26.000000,1.000000,90.000000,1.000000,1.000000,0.0,1.000000,1.0,1.000000


In [55]:
def check_data_quality(data,
                       freq_treshold=0.95,
                       nunique_threshold=0.95,
                       null_threshold=0):
    """Проводит поиск дублей, пропусков и неинформативных признаков и выводит результат в консоль.
    
    Неинформативным считается признак с долей уникальных значений или повторов выше установленного порога.
    
    Parameters
    ----------
        data : DataFrame
            Датафрейм для анализа
        freq_treshold : float
            Порог масимальной частоты встречаемости признака
        nunique_treshhold : float
            Порог максимальной уникальности признака
        null_threshold : float
            Порог максимального отстутсвия признака
    """
    
    # Поиск дублей
    try:
        print(f'Число найденных дублей: {
            data.duplicated().value_counts().loc[True]
            }\n')
    except KeyError:
        print('Количество дублей: 0\n')

    # Поиск пропущенных значений
    cols_null_percent = data.isnull().mean()
    cols_with_null = cols_null_percent[cols_null_percent > null_threshold]\
        .sort_values(ascending=False)
    if cols_with_null.shape[0]: display(cols_with_null)
    print(f'Количество признаков с пустыми значениями: {cols_with_null.shape[0]}\n')

    # Поиск неинформативных признаков
    low_information_cols = []
    bad_feat_flag = False
    # цикл по всем столбцам
    for col in data.columns:
        #наибольшая относительная частота в признаке
        top_freq = data[col].value_counts(normalize=True).max()
        #доля уникальных значений от размера признака
        nunique_ratio = data[col].nunique() / data[col].count()
        # сравниваем наибольшую частоту с порогом
        if top_freq > freq_treshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {top_freq:.2%} одинаковых значений')
        # сравниваем долю уникальных значений с порогом
        if nunique_ratio > nunique_threshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {nunique_ratio:.2%} уникальных значений')
    if not bad_feat_flag:
        print('Неинформативных признаков не найдено.')

In [56]:
print('Проверка тренировочной выборки:'+'\n'+'-' * 30)
check_data_quality(work_df)

Проверка тренировочной выборки:
------------------------------
Количество дублей: 0



normoblast      0.998395
prolym          0.994864
normobl_rel     0.994222
promyelo        0.993900
plasma          0.990369
ret_abs         0.987159
ret_rel         0.983307
plcr            0.973997
noclass_abs     0.963082
blasty          0.955377
macrocytos      0.954735
microcytos      0.946067
normobl_abs     0.942536
noclass_rel     0.923917
hypochromia     0.921348
yunye           0.906581
myelo           0.903692
mxd_rel         0.852006
mxd_abs         0.835634
pdw             0.791332
rdv_cv          0.780738
rdv_sd          0.779775
pal             0.770465
seg             0.770144
soe             0.757624
poikilocytos    0.748315
ba_abs          0.746067
mo_rel          0.744783
ba_rel          0.743178
cp              0.742536
eo_rel          0.742215
rdw             0.741573
eo_abs          0.721669
pct             0.712360
mo_abs          0.703050
mpv             0.692456
anisocytos      0.688604
ne_abs          0.588764
ne_rel          0.587159
ly_rel          0.543178


Количество признаков с пустыми значениями: 49

study_ID: 100.00% уникальных значений
normobl_rel: 100.00% одинаковых значений
macrocytos: 100.00% одинаковых значений
normoblast: 100.00% одинаковых значений
